# Judge on teasers made from a page snapshot (task YL-004B, lab)

The judge reads the **text of the page** (the snapshot from `evidence_fetch`) as its record and checks a teaser against it. No hand-written facts.
You upload three files: the pack made from the snapshot, and one or two teaser texts. The cells run the free checks first, then the model.

**Before you start:** the Colab secret `GEMINI_API_KEY` must exist (key icon, "Notebook access" on). The free Gemini tier may be used by Google for training, so use it only for public page text, never for private data.

### Step 1. Get the code (branch `lab-step3`) and install the packages
If the folder already exists from an earlier session, the cell updates it, so you always run the newest rules.

In [ ]:
import os

REPO_URL = "https://github.com/Yul04ek/ai-content-claim-check.git"
BRANCH = "lab-step3"            # the teaser check lives on this branch, not on main
FOLDER = "/content/claim-check"

if not os.path.exists(FOLDER):
    # A private repository needs a token. A public one works without it.
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
        clone_url = REPO_URL.replace("https://", f"https://{token}@")
    except Exception:
        clone_url = REPO_URL
    !git clone -q -b {BRANCH} {clone_url} {FOLDER}
    !git -C {FOLDER} remote set-url origin {REPO_URL}   # do not keep the token in the repo settings
else:
    !git -C {FOLDER} fetch -q origin {BRANCH}
    !git -C {FOLDER} checkout -q {BRANCH}
    !git -C {FOLDER} pull -q origin {BRANCH}
%cd {FOLDER}
!git log --oneline -1
!pip install -q -r requirements.txt

### Step 2. Connect the model
Same test as in the other notebook. It should print `ready`. If the model is overloaded (503), the cell waits and tries again.

In [ ]:
import os
from google.colab import userdata

# Read the key from Colab Secrets
try:
    os.environ["API_KEY"] = userdata.get("GEMINI_API_KEY")
except Exception:
    raise RuntimeError(
        "Secret GEMINI_API_KEY not found. Add it with the key icon in the left sidebar, "
        "switch on 'Notebook access', and run this cell again."
    )

os.environ["BASE_URL"] = "https://generativelanguage.googleapis.com/v1beta/openai/"  # Gemini through the OpenAI-compatible API
os.environ["MODEL_NAME"] = "gemini-3.5-flash-lite"   # the model that worked in the research agent; change if needed

# Send one short message to check that the key and the model work. Waits and tries again when the model is busy.
import time
from openai import OpenAI, APIStatusError

client = OpenAI(base_url=os.environ["BASE_URL"], api_key=os.environ["API_KEY"]).with_options(timeout=30, max_retries=0)
for attempt in range(5):
    try:
        reply = client.chat.completions.create(
            model=os.environ["MODEL_NAME"],
            messages=[{"role": "user", "content": "Reply with the single word: ready"}],
        )
        print(reply.choices[0].message.content)
        break
    except APIStatusError as error:
        # 503 = the model is overloaded, 429 = per-minute limit. Both are temporary (404 would mean a wrong model name).
        if error.status_code in (429, 500, 502, 503, 504) and attempt < 4:
            wait = 20 * (attempt + 1)
            print(f"Model answered {error.status_code}, waiting {wait} s ...")
            time.sleep(wait)
        else:
            raise

### Step 3. Upload the files
Run the cell and choose the files (one zip with all of them, or the files one at a time): the pack (`pack_....json`) and the teaser texts (`.txt`). Files whose name contains `pack` are used as packs. With several packs, a teaser gets the pack whose name has the company word of the teaser name (`teaser_Tuta_9nov.txt` -> `pack_Tuta_....json`).

In [ ]:
%cd /content/claim-check
import os
from google.colab import files

os.makedirs("lab_upload", exist_ok=True)
uploaded = files.upload()   # can be run several times, one file each time
import zipfile
for name, data in uploaded.items():
    path = os.path.join("lab_upload", name)
    open(path, "wb").write(data)
    if name.lower().endswith(".zip"):      # a zip with packs and teasers: unpack the files flat into lab_upload
        with zipfile.ZipFile(path) as z:
            for member in z.namelist():
                if not member.endswith("/") and not os.path.basename(member).startswith((".", "_")):
                    open(os.path.join("lab_upload", os.path.basename(member)), "wb").write(z.read(member))
        os.remove(path)

for n in os.listdir("lab_upload"):
    if " (" in n:          # Colab adds " (1)" to a repeated upload: remove such copies
        os.remove(os.path.join("lab_upload", n))

names = sorted(os.listdir("lab_upload"))
PACKS = [n for n in names if "pack" in n and n.endswith(".json")]
TEASERS = [n for n in names if n.endswith(".txt")]
assert PACKS, "No pack file found: its name must contain 'pack' and end with .json"


def pack_for(teaser):
    """The pack of a teaser. One pack: that one. Several packs: the one whose name has the company word of the teaser name
    (teaser_Tuta_9nov.txt -> pack_Tuta_snapshot_SIM_003.json)."""
    if len(PACKS) == 1:
        return PACKS[0]
    words = [w for w in os.path.splitext(teaser)[0].lower().split("_") if w != "teaser"]
    found = [p for p in PACKS if words and words[0] in p.lower()]
    assert len(found) == 1, f"No single pack for {teaser}: found {found} among {PACKS}"
    return found[0]


for t in TEASERS:
    print(t, "->", pack_for(t))

### Step 4. Free checks (no model call)
Offline tests, then the deterministic rules on every teaser. A dry run is never a verdict.

In [ ]:
%cd /content/claim-check
!python tests/test_teaser_check.py | tail -1
!python tests/test_evidence_fetch.py | tail -1
for t in TEASERS:
    print("\n#####", t)
    !python -m teaser_check "lab_upload/{pack_for(t)}" "lab_upload/{t}" --dry-run --out /tmp/dry.csv

### Step 5. The judge (model)
One call per teaser. If the model is overloaded (503) the helper waits and tries again. The result of each teaser is printed, and every row is saved in `results/`.

In [ ]:
%cd /content/claim-check
import os, datetime
stamp = f"{datetime.datetime.now():%Y-%m-%d_%H-%M}"
RUBRIC = "skills/experiments/n1.md"   # rubric of the judge; None = the old default rubric (skills/teaser-judge.md, made for the old format)
rubric_name = os.path.splitext(os.path.basename(RUBRIC))[0] if RUBRIC else "t1"
rubric_flag = f"--rubric {RUBRIC}" if RUBRIC else ""
for t in TEASERS:
    out = f"results/lab_{os.path.splitext(t)[0]}_{os.environ['MODEL_NAME']}_{rubric_name}_{stamp}.csv"
    print("\n#####", t)
    !python -m teaser_check "lab_upload/{pack_for(t)}" "lab_upload/{t}" --out {out} {rubric_flag}

### Step 6. All rows that are not plainly Supported

In [ ]:
%cd /content/claim-check
import glob, pandas as pd
pd.set_option("display.max_colwidth", 110)
pd.set_option("display.width", 220)
for f in sorted(glob.glob(f"results/lab_*_{stamp}.csv")):
    df = pd.read_csv(f)
    print("\n", f, "|", df["verdict"].value_counts(dropna=False).to_dict())
    display(df[df["verdict"] != "Supported"][["kind", "text", "verdict", "finding", "reason"]])

### Step 7. Save the results
Everything in Colab is deleted when the session ends. This downloads the `results` folder.

In [ ]:
%cd /content/claim-check
!zip -qr /content/teaser_lab_results.zip results
from google.colab import files
files.download("/content/teaser_lab_results.zip")